## Routed Recharge ##

**This notebook takes a recharge field from the ecohydrology model and accumulates recharge downstream. 
It then divides the cumulative recharge flow to the local drainage area, calculating an upslope averaged recharge field**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline
import pylab

from landlab.plot.imshow import imshow_grid
from landlab.plot.imshow import imshow_grid_at_node

# read ESRI
from landlab.io import read_esri_ascii, write_esri_ascii

from landlab.grid.mappers import map_node_to_cell


from landlab.components.flow_accum import find_drainage_area_and_discharge


from landlab.components import FlowAccumulator, SinkFiller, SinkFillerBarnes


import time
st = time.time()

In [ ]:
# read the DEM and create a raster model grid  
(grid,Z)=read_esri_ascii('landslide_run_dem.asc', name='topographic__elevation')  

# here we set nodata to closed boundary
grid.set_nodata_nodes_to_closed(Z, -9999.) 

# find the lowest point on our DEM?       
outlet_id = grid.core_nodes[np.argmin(grid.at_node['topographic__elevation'][grid.core_nodes])]      

# set the lowest point as the outlet, make all the nodes above this point core nodes
grid.set_watershed_boundary_condition_outlet_id(outlet_id, Z)    

print("Outlet ID=", outlet_id)                                        # print outlet id number
print("Outlet elevation=",grid.at_node['topographic__elevation'][outlet_id])        # print elevation of outlet node
print("Min elevation of core nodes=", np.min(grid.at_node['topographic__elevation'][grid.core_nodes])) # print minimum elevation of core nodes
print("Max elevation of core nodes=", np.max(grid.at_node['topographic__elevation'][grid.core_nodes])) # print maximum elevation of co

In [ ]:
(RG,Recharge)=read_esri_ascii('IMERG_Recharge_December_2018.asc', name='Recharge') #cm

In [ ]:
for i in range(len(Recharge)):
     if Recharge[i] <= 0:
        Recharge[i] = 0.0001

#Recharge_rate=Recharge
water__unit_flux_in=Recharge

In [ ]:
_=grid.add_field('node', 'water__unit_flux_in', water__unit_flux_in, at='node', clobber=True)
imshow_grid_at_node(grid, 'water__unit_flux_in', cmap='Greens' )

In [ ]:
sfb = SinkFillerBarnes(grid,'topographic__elevation', method='D8', fill_flat = True,
                      ignore_overfill = False)
sfb.run_one_step()

In [ ]:
fa = FlowAccumulator(grid,
                     surface='topographic__elevation',
                     flow_director='FlowDirectorD8',
                     depression_finder='DepressionFinderAndRouter')

In [ ]:
(da, q) = fa.accumulate_flow()

In [ ]:
grid.at_node['drainage_area'][np.where(da==0)] = (grid.dx*grid.dy)  

In [ ]:
_ = grid.add_field('routed_recharge', (grid.at_node['surface_water__discharge'] / grid.at_node['drainage_area']), at='node', clobber=True)

In [ ]:
imshow_grid_at_node(grid, 'routed_recharge', cmap='Greens' )

In [ ]:
_ = grid.add_field('diff_recharge', (grid.at_node['routed_recharge'] - grid.at_node['water__unit_flux_in']), at='node', clobber=True)

In [ ]:
imshow_grid_at_node(grid, 'diff_recharge', cmap='jet' )

In [ ]:
write_esri_ascii('IMERG_Recharge_December_2018_routed.asc', grid, 'routed_recharge', clobber=True)